# 🚀 LidarScan Studio — Free Google Colab GPU Splat Trainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zaiahgaming/lidarscan-studio/blob/main/colab_trainer.ipynb)

Train high-quality **3D Gaussian Splats** from your iPhone LiDAR captures in **2–5 minutes** using a **free NVIDIA T4 GPU** on Google Colab.

### Workflow:
1. **Runtime -> Change runtime type -> T4 GPU** (if not already selected)
2. Run **Cell 1** (Check GPU & Install Trainer)
3. Run **Cell 2** (Upload your `<capture>.lidarscan.zip` or fetch via URL)
4. Run **Cell 3** (Train Gaussian Splats at high speed)
5. Run **Cell 4** (Download the generated `.splat` and import it directly into LidarScan Studio!)

In [ ]:
# @title 1. Verify GPU and Install Tools
!nvidia-smi

import os, sys, shutil, urllib.request, subprocess

# Download latest Brush Linux release (works with CUDA / Vulkan)
BRUSH_URL = "https://github.com/ArthurBrussee/brush/releases/download/v0.3.0/brush-app-x86_64-unknown-linux-gnu.tar.xz"
print("📥 Downloading Brush 3D Gaussian Splatting engine...")
urllib.request.urlretrieve(BRUSH_URL, "brush.tar.xz")
!tar -xf brush.tar.xz
os.chmod("brush-app-x86_64-unknown-linux-gnu/brush_app", 0o755)
print("✅ Brush installed successfully!")

In [ ]:
# @title 2. Upload Capture Bundle (.lidarscan.zip)
from google.colab import files
import zipfile, glob

os.makedirs("workspace/dataset", exist_ok=True)
print("📤 Please select and upload your .lidarscan.zip file:")
uploaded = files.upload()

zip_filename = None
for fn in uploaded.keys():
    if fn.endswith(".zip"):
        zip_filename = fn
        break

if not zip_filename:
    raise ValueError("No zip file uploaded!")

print(f"📦 Unpacking {zip_filename}...")
with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall("workspace/dataset")

# Locate directory with transforms.json
dataset_dir = "workspace/dataset"
if not os.path.exists(os.path.join(dataset_dir, "transforms.json")):
    matches = glob.glob("workspace/dataset/**/transforms.json", recursive=True)
    if matches:
        dataset_dir = os.path.dirname(matches[0])

print(f"✅ Capture ready at: {dataset_dir}")

In [ ]:
# @title 3. Train Gaussian Splats on Colab GPU
TOTAL_STEPS = 7000  # @param {type:"integer"}
MAX_RESOLUTION = 1280  # @param {type:"integer"}
MAX_SPLATS = 600000  # @param {type:"integer"}

os.makedirs("workspace/output", exist_ok=True)

cmd = [
    "./brush-app-x86_64-unknown-linux-gnu/brush_app",
    dataset_dir,
    "--total-steps", str(TOTAL_STEPS),
    "--max-splats", str(MAX_SPLATS),
    "--max-resolution", str(MAX_RESOLUTION),
    "--export-every", str(TOTAL_STEPS),
    "--export-path", "workspace/output",
    "--export-name", "trained_splat.ply"
]

print(f"🔥 Training Gaussian Splats ({TOTAL_STEPS} steps)... This will take ~2-4 minutes.")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Training failed with code {process.returncode}")

print("\n🎉 Training completed successfully!")

In [ ]:
# @title 4. Convert to Compact .splat and Download
import numpy as np
from google.colab import files

def convert_ply_to_splat(ply_path, splat_path):
    SH_C0 = 0.28209479177387814
    with open(ply_path, "rb") as f:
        num_vertices = 0
        properties = []
        while True:
            line = f.readline().decode("utf-8", errors="ignore").strip()
            if line.startswith("element vertex"):
                num_vertices = int(line.split()[-1])
            elif line.startswith("property"):
                parts = line.split()
                properties.append((parts[2], parts[1]))
            elif line == "end_header":
                break
        
        type_map = {"float": "<f4", "float32": "<f4", "double": "<f8", "uchar": "u1", "uint8": "u1", "int": "<i4"}
        dtype_list = [(p_name, type_map.get(p_type, "<f4")) for p_name, p_type in properties]
        data = np.fromfile(f, dtype=np.dtype(dtype_list), count=num_vertices)

    pos_x, pos_y, pos_z = data["x"].astype(np.float32), data["y"].astype(np.float32), data["z"].astype(np.float32)
    s0, s1, s2 = np.exp(data["scale_0"]), np.exp(data["scale_1"]), np.exp(data["scale_2"])
    rgb_r = np.clip((0.5 + SH_C0 * data["f_dc_0"]) * 255.0, 0, 255).astype(np.uint8)
    rgb_g = np.clip((0.5 + SH_C0 * data["f_dc_1"]) * 255.0, 0, 255).astype(np.uint8)
    rgb_b = np.clip((0.5 + SH_C0 * data["f_dc_2"]) * 255.0, 0, 255).astype(np.uint8)
    opacity = 1.0 / (1.0 + np.exp(-data["opacity"]))
    rgb_a = np.clip(opacity * 255.0, 0, 255).astype(np.uint8)

    qw, qx, qy, qz = data["rot_0"], data["rot_1"], data["rot_2"], data["rot_3"]
    q_norm = np.maximum(np.sqrt(qw**2 + qx**2 + qy**2 + qz**2), 1e-8)
    rot_0 = np.clip((qw/q_norm * 128.0) + 128.0, 0, 255).astype(np.uint8)
    rot_1 = np.clip((qx/q_norm * 128.0) + 128.0, 0, 255).astype(np.uint8)
    rot_2 = np.clip((qy/q_norm * 128.0) + 128.0, 0, 255).astype(np.uint8)
    rot_3 = np.clip((qz/q_norm * 128.0) + 128.0, 0, 255).astype(np.uint8)

    splat_dtype = np.dtype([("pos", "<f4", (3,)), ("scale", "<f4", (3,)), ("color", "u1", (4,)), ("rot", "u1", (4,))])
    out = np.empty(num_vertices, dtype=splat_dtype)
    out["pos"][:, 0], out["pos"][:, 1], out["pos"][:, 2] = pos_x, pos_y, pos_z
    out["scale"][:, 0], out["scale"][:, 1], out["scale"][:, 2] = s0, s1, s2
    out["color"][:, 0], out["color"][:, 1], out["color"][:, 2], out["color"][:, 3] = rgb_r, rgb_g, rgb_b, rgb_a
    out["rot"][:, 0], out["rot"][:, 1], out["rot"][:, 2], out["rot"][:, 3] = rot_0, rot_1, rot_2, rot_3

    with open(splat_path, "wb") as fo:
        out.tofile(fo)
    print(f"✅ Successfully wrote {num_vertices:,} splats to {splat_path}")

ply_file = "workspace/output/trained_splat.ply"
splat_file = "workspace/output/trained_splat.splat"
convert_ply_to_splat(ply_file, splat_file)

print("📥 Triggering download of trained_splat.splat to your computer...")
files.download(splat_file)
print("🎉 Done! In LidarScan Studio, open your capture and click 'Import Splat' to view in 3D!")